# Curvilinear Wave Equation

The Cartesian wave-equation notebooks introduced the first-order wave system and the generated-project workflow. This notebook uses the same equation on curvilinear coordinates by inserting reference-metric inverse metrics and contracted Christoffel terms.

## Table of Contents

1. [Reference-metric form](#Reference-metric-form)
2. [Imports and generator discovery](#Imports-and-generator-discovery)
3. [Spherical right-hand side](#Spherical-right-hand-side)
4. [Cartesian consistency check](#Cartesian-consistency-check)
5. [Precompute-enabled curvilinear path](#Precompute-enabled-curvilinear-path)
6. [Generated project workflow](#Generated-project-workflow)
7. [Related notebooks](#Related-notebooks)

## Reference-Metric Form

The first-order wave system remains

$$
\partial_t u = v,
$$

$$
\partial_t v =
c^2\left(\hat{g}^{ij}\partial_i\partial_j u
- \hat{\Gamma}^i\partial_i u\right).
$$

Here $\hat{g}^{ij}$ is the inverse reference metric and $\hat{\Gamma}^i=\hat{g}^{jk}\hat{\Gamma}^{i}_{jk}$ is the contracted reference-metric connection. NRPy represents the derivative variables symbolically, then the generated project lowers those symbols to finite-difference data accesses.

## Imports and Generator Discovery

Discovery uses `importlib.util.find_spec` so the packaged generator is not imported at notebook startup.

In [ ]:
import importlib.util
import shutil
import subprocess
import sys
import tempfile
from pathlib import Path

import nrpy
import nrpy.grid as grid
import nrpy.params as par
import sympy as sp
from nrpy.equations.wave_equation.WaveEquation_RHSs import WaveEquation_RHSs
from nrpy.equations.wave_equation.WaveEquationCurvilinear_RHSs import (
    WaveEquationCurvilinear_RHSs,
)


print("Imported nrpy from:", nrpy.__file__)

generator_name = "nrpy.examples.wave_equation_curvilinear"
generator_spec = importlib.util.find_spec(generator_name)
print("Generator discovered:", generator_spec is not None)

In [ ]:
notebook_wave_gridfunctions = set()


def construct_wave_rhs(factory, *args, **kwargs):
    before = set(grid.glb_gridfcs_dict)
    rhs = factory(*args, **kwargs)
    notebook_wave_gridfunctions.update(set(grid.glb_gridfcs_dict) - before)
    return rhs


def clear_notebook_wave_gridfunctions():
    for name in sorted(notebook_wave_gridfunctions):
        grid.glb_gridfcs_dict.pop(name, None)
    notebook_wave_gridfunctions.clear()


def compact(expr, width=180):
    text = sp.sstr(expr)
    return text if len(text) <= width else text[: width - 3] + "..."


print("Wave gridfunctions before RHS construction:", sorted(grid.glb_gridfcs_dict))

## Spherical Right-Hand Side

In spherical coordinates, the curvilinear module produces the radial, polar, and azimuthal second-derivative terms plus the reference-metric first-derivative terms.

In [ ]:
clear_notebook_wave_gridfunctions()
spherical_rhs = construct_wave_rhs(
    WaveEquationCurvilinear_RHSs,
    CoordSystem="Spherical", enable_rfm_precompute=False
)

symbols = {symbol.name: symbol for symbol in spherical_rhs.vv_rhs.free_symbols}
wavespeed = symbols["wavespeed"]
xx0 = symbols["xx0"]
xx1 = symbols["xx1"]
uu_dD0 = symbols["uu_dD0"]
uu_dD1 = symbols["uu_dD1"]
uu_dDD00 = symbols["uu_dDD00"]
uu_dDD11 = symbols["uu_dDD11"]
uu_dDD22 = symbols["uu_dDD22"]

expected_spherical_vv_rhs = wavespeed**2 * (
    uu_dDD00
    + uu_dDD11 / xx0**2
    + uu_dDD22 / (xx0**2 * sp.sin(xx1) ** 2)
    + 2 * uu_dD0 / xx0
    + uu_dD1 * sp.cos(xx1) / (xx0**2 * sp.sin(xx1))
)
spherical_residual = sp.trigsimp(
    sp.simplify(spherical_rhs.vv_rhs - expected_spherical_vv_rhs)
)

print("uu_rhs =", spherical_rhs.uu_rhs)
print("vv_rhs =", compact(spherical_rhs.vv_rhs))
print("Spherical RHS residual:", spherical_residual)

## Cartesian Consistency Check

With Cartesian coordinates, the curvilinear class reduces to the Cartesian wave-equation right-hand side.

In [ ]:
clear_notebook_wave_gridfunctions()
cartesian_rhs = construct_wave_rhs(WaveEquation_RHSs)
clear_notebook_wave_gridfunctions()
curvilinear_cartesian_rhs = construct_wave_rhs(
    WaveEquationCurvilinear_RHSs,
    CoordSystem="Cartesian", enable_rfm_precompute=False
)
cartesian_residual = sp.simplify(
    curvilinear_cartesian_rhs.vv_rhs - cartesian_rhs.vv_rhs
)

print("Cartesian vv_rhs residual:", cartesian_residual)

## Precompute-Enabled Curvilinear Path

The packaged curvilinear generator uses a stretched cylindrical coordinate system and RFM precompute. The symbolic right-hand side then references compact precomputed factors instead of repeatedly expanding the coordinate map.

In [ ]:
clear_notebook_wave_gridfunctions()
par.set_parval_from_str("CoordSystem_to_register_CodeParameters", "SinhCylindrical")
precompute_rhs = construct_wave_rhs(
    WaveEquationCurvilinear_RHSs,
    CoordSystem="SinhCylindrical", enable_rfm_precompute=True
)

print("uu_rhs =", precompute_rhs.uu_rhs)
print("vv_rhs string length:", len(str(precompute_rhs.vv_rhs)))
print("vv_rhs excerpt:", compact(precompute_rhs.vv_rhs))

## Generated Project Workflow

The generator command is:

```bash
python -m nrpy.examples.wave_equation_curvilinear --disable_intrinsics
```

It is run from a temporary workspace so an existing project directory is not overwritten. CUDA and heavier parallelization variants are outside this notebook's default path.

In [ ]:
workspace_manager = tempfile.TemporaryDirectory(
    prefix=".nrpy-curvi-wave-", dir=Path.cwd()
)
workspace_path = Path(workspace_manager.name)
project_path = workspace_path / "project" / "wave_equation_curvilinear"
generator_env = {"HOME": str(workspace_path)}

print("Temporary workspace created:", workspace_path.name)
print("Project directory exists before generation:", project_path.exists())

In [ ]:
generation_completed = False
if generator_spec is None:
    print(f"Generator not found: {generator_name}")
else:
    generate = subprocess.run(
        [sys.executable, "-m", generator_name, "--disable_intrinsics"],
        cwd=workspace_path,
        env=generator_env,
        check=False,
        capture_output=True,
        text=True,
        timeout=300,
    )
    generation_completed = generate.returncode == 0
    print("Generator return code:", generate.returncode)
    if generate.stdout.strip():
        print("\n".join(generate.stdout.splitlines()[:12]))
    if generate.returncode != 0 and generate.stderr.strip():
        print("Generator stderr excerpt:")
        print("\n".join(generate.stderr.splitlines()[:12]))

if project_path.exists():
    expected_paths = [
        project_path / "Makefile",
        project_path / "wave_equation_curvilinear.par",
        project_path / "BHaH_defines.h",
        project_path / "MoL",
        project_path / "diagnostics",
        project_path / "SinhCylindrical",
    ]
    print("Expected path inventory:")
    for path in expected_paths:
        print(" ", path.relative_to(project_path), "exists:", path.exists())
    generated_files = sorted(
        path.relative_to(project_path)
        for path in project_path.rglob("*")
        if path.is_file()
    )
    print("Generated file count:", len(generated_files))
    print("First generated files:")
    for relpath in generated_files[:12]:
        print(" ", relpath)
else:
    print("Generated project directory is not present.")

## Build and Run

Build and run steps require `make` and a C compiler. If the tools are unavailable, the generated project can be built later from its project directory.

In [ ]:
make_tool = shutil.which("make")
compiler_tool = shutil.which("cc") or shutil.which("gcc") or shutil.which("clang")
makefile_path = project_path / "Makefile"
can_build = (
    generation_completed
    and makefile_path.exists()
    and make_tool is not None
    and compiler_tool is not None
)

print("make available:", make_tool is not None)
print("C compiler available:", compiler_tool is not None)

build_completed = False
run_completed = False
if not project_path.exists():
    print("Build skipped because no generated project is available.")
elif not generation_completed or not makefile_path.exists():
    print("Build skipped because generation did not produce a Makefile.")
elif not can_build:
    print("Build skipped. From the generated project directory, run: make -j2")
    print("Then run: ./wave_equation_curvilinear")
    print("For a convergence rerun, run: ./wave_equation_curvilinear 2.0")
else:
    build = subprocess.run(
        ["make", "-j2"],
        cwd=project_path,
        check=False,
        capture_output=True,
        text=True,
        timeout=300,
    )
    build_completed = build.returncode == 0
    print("Build return code:", build.returncode)
    if build.stdout.strip():
        print("\n".join(build.stdout.splitlines()[:16]))
    if build.returncode != 0 and build.stderr.strip():
        print("Build stderr excerpt:")
        print("\n".join(build.stderr.splitlines()[:16]))

    executable = project_path / "wave_equation_curvilinear"
    print("Executable exists:", executable.exists())
    if build_completed and executable.exists():
        for args in ([], ["2.0"]):
            label = "default" if not args else "convergence factor 2.0"
            run = subprocess.run(
                [f"./{executable.name}", *args],
                cwd=project_path,
                check=False,
                capture_output=True,
                text=True,
                timeout=180,
            )
            print(f"Run ({label}) return code:", run.returncode)
            if run.stdout.strip():
                print("\n".join(run.stdout.splitlines()[:12]))
            if run.returncode != 0 and run.stderr.strip():
                print("Run stderr excerpt:")
                print("\n".join(run.stderr.splitlines()[:12]))
            run_completed = run_completed or run.returncode == 0

In [ ]:
if project_path.exists():
    diagnostics = sorted(project_path.glob("out0d-conv_factor*.txt"))
    if diagnostics:
        for diagnostic in diagnostics:
            print("Diagnostic file:", diagnostic.name)
            print(
                "\n".join(
                    diagnostic.read_text(
                        encoding="utf-8", errors="replace"
                    ).splitlines()[:10]
                )
            )
    elif run_completed:
        print("No convergence diagnostic files were found after successful runs.")
    else:
        print("No convergence diagnostics are available.")
else:
    print("No generated project is available for diagnostic inspection.")

## Related Notebooks

[Reference-metric applications](../4-curvilinear/reference_metric_applications.ipynb) explains the coordinate maps and precompute factors used here. [Basis transforms](../4-curvilinear/basis_transforms.ipynb) shows how the stored Jacobians act on vectors and tensors.